Importing a library needed

In [ ]:
import sqlite3
import pandas as pd
import sqlalchemy
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


Connecting a database

In [25]:
conn = sqlite3.connect("../superstore.sqlite")

Combining tables (Denormalizing the table)

In [26]:
df = pd.read_sql_query(
    """
    SELECT
        oi.row_id,
        oi.order_key,
        oi.sales,
        oi.quantity,
        oi.discount,
        oi.profit,
        oi.shipping_cost,
        o.order_date,
        o.ship_date,
        o.ship_mode,
        o.order_priority,
        dc.customer_id,
        dc.segment,
        dp.category,
        dp.sub_category,
        dp.product_name
    FROM order_items oi 
    JOIN dim_products dp ON oi.product_id = dp.product_id
    JOIN orders o ON oi.order_key = o.order_key 
    JOIN dim_customers dc ON o.customer_id = dc.customer_id
    """,
    conn,
)
df.head()

,row_id,order_key,sales,quantity,discount,profit,shipping_cost,order_date,ship_date,ship_mode,order_priority,customer_id,segment,category,sub_category,product_name
0,1,MX-2014-143658_SC-205753,13.0,3,0.0,4.56,1.033,2014-10-02 00:00:00,2014-10-06 00:00:00,Standard Class,Medium,SC-205753,Consumer,Office Supplies,Labels,"Hon File Folder Labels, Adjustable"
1,2,MX-2012-155047_KW-165703,252.0,8,0.0,90.72,13.449,2012-10-15 00:00:00,2012-10-20 00:00:00,Standard Class,Medium,KW-165703,Consumer,Furniture,Furnishings,"Tenex Clock, Durable"
2,3,MX-2012-155047_KW-165703,193.0,2,0.0,54.08,9.627,2012-10-15 00:00:00,2012-10-20 00:00:00,Standard Class,Medium,KW-165703,Consumer,Furniture,Bookcases,"Ikea 3-Shelf Cabinet, Mobile"
3,4,MX-2012-155047_KW-165703,35.0,4,0.0,4.96,1.371,2012-10-15 00:00:00,2012-10-20 00:00:00,Standard Class,Medium,KW-165703,Consumer,Office Supplies,Binders,"Cardinal Binder, Clear"
4,5,MX-2012-155047_KW-165703,72.0,2,0.0,11.44,3.787,2012-10-15 00:00:00,2012-10-20 00:00:00,Standard Class,Medium,KW-165703,Consumer,Office Supplies,Art,"Sanford Canvas, Water Color"


Cleaning up data

In [27]:
# Deduplicate
dupes = df.duplicated().sum()
if dupes > 0:
    df = df.drop_duplicates().reset_index(drop=True)

# Text Normalization
cols_to_be_text_normalized = [
    "ship_mode",
    "order_priority",
    "segment",
    "category",
    "sub_category",
    "product_name",
]

for col in cols_to_be_text_normalized:
    df[col] = df[col].astype(str).str.lower().str.strip()

# Date
cols_to_be_date_formatted = [
    "order_date",
    "ship_date",
]

for col in cols_to_be_date_formatted:
    df[col] = pd.to_datetime(df[col])

# Add new features
df["is_profitable"] = (df["profit"] > 0).astype(int)

Split the feature and target

In [28]:
feature_cols = [
    "discount",
    "shipping_cost",
    "ship_mode",
    "segment",
    "category",
    "sub_category",
]

X = df[feature_cols]
y = df["is_profitable"]

Split the data to training and testing data

In [29]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=131205, test_size=0.2, stratify=y)

Preprocessing the data

In [30]:
numeric_features = ["discount", "shipping_cost"]
categorical_features = ["ship_mode", "segment", "category", "sub_category"]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
])

Train the model

In [31]:
set_config(display="diagram")

model = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", LogisticRegression(class_weight="balanced", random_state=131205))
])

model

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['discount',
                                                   'shipping_cost']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['ship_mode', 'segment',
                                                   'category',
                                                   'sub_category'])])),
                ('model',
                 LogisticRegression(class_weight='balanced',
                                    random_state=131205))])

Check the model

In [32]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.80      0.85      0.83      2642
           1       0.95      0.93      0.94      7616

    accuracy                           0.91     10258
   macro avg       0.87      0.89      0.88     10258
weighted avg       0.91      0.91      0.91     10258

